# 3.4 Dynamic models

`@wrap_model_call` can also swap the **chat model**. Short threads use a cheap/fast model; long threads use a larger-context model.

Needs `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` (Claude for the long-conversation path).

## Setup

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

`request.messages` is a shortcut for `request.state["messages"]`. If `len > 10`, override to Claude; otherwise `gpt-5-nano`.

`create_agent(model=...)` is only the default before middleware runs.

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("gpt-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

## Short conversation (≤10 messages)

`response_metadata["model_name"]` should be the nano / GPT model.

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I can’t water it remotely, so I didn’t water it today. If you’d like, I can set up a daily watering reminder or assign the task to someone on the team. What schedule should we use (e.g., every weekday at 9:30 AM)? If you want, I can also draft a quick care note for the plant with how often it needs water.


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


## Long conversation (>10 messages)

Eleven messages in the list → Claude. Check `model_name` again.

In [8]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CeyJfbjhapSgYPmiBhvrY'}

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])